# Lab 10 — Spark SQL: consultas interativas

## Objetivo

Este laboratório utiliza o PySpark em modo local para consultar a camada Silver.

A mesma agregação será executada por Spark SQL e pela API de DataFrame, demonstrando duas formas de interação com o mesmo motor. Também será avaliado o uso de cache em memória e realizada uma análise de fraude por dia da semana.

O processamento ocorre localmente, utilizando os núcleos do computador, e não representa um cluster distribuído com várias máquinas.

In [1]:
import os
import sys
import time
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
PySpark: 4.2.0
JAVA_HOME: C:\BigData\java\jdk-17.0.20.1+1


In [2]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

pasta_projeto = Path.cwd().resolve()

if not (pasta_projeto / "dados" / "silver").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "silver").exists():
            pasta_projeto = pasta_pai
            break

arquivo_silver = (
    pasta_projeto
    / "dados"
    / "silver"
    / "transactions_enriched.parquet"
)

assert arquivo_silver.exists(), "Arquivo Silver não encontrado."

# O Spark no Windows trabalha melhor com barras normais.
caminho_silver_spark = arquivo_silver.as_posix()

print("Silver:", caminho_silver_spark)

Silver: C:/BigData/bigdata-curso-gabriel/dados/silver/transactions_enriched.parquet


In [3]:
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab10_spark")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("Aplicação:", spark.sparkContext.appName)
print("Modo:", spark.sparkContext.master)

c:\BigData\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark: 4.2.0
Aplicação: lab10_spark
Modo: local[2]


In [4]:
df = spark.read.parquet(caminho_silver_spark)

print("Quantidade de transações:", df.count())
print("Quantidade de colunas:", len(df.columns))

Quantidade de transações: 100000
Quantidade de colunas: 15


In [5]:
df.printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_timestamp: timestamp_ntz (nullable = true)
 |-- status: string (nullable = true)
 |-- risk_score: double (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- segment: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- year: long (nullable = true)
 |-- month: long (nullable = true)
 |-- day: long (nullable = true)
 |-- day_of_week: long (nullable = true)
 |-- amount_band: string (nullable = true)



In [6]:
df.show(5, truncate=False)

+--------------+-----------+------------------+----------------+---------------------+--------+------------------+--------+---------+------------+----+-----+---+-----------+-----------+
|transaction_id|customer_id|amount            |transaction_type|transaction_timestamp|status  |risk_score        |is_fraud|segment  |credit_score|year|month|day|day_of_week|amount_band|
+--------------+-----------+------------------+----------------+---------------------+--------+------------------+--------+---------+------------+----+-----+---+-----------+-----------+
|31732         |9120       |101.42404535018144|compra          |2023-03-25 00:00:00  |approved|7.288249051509188 |false   |Standard |673         |2023|3    |25 |6          |medio      |
|31747         |6958       |110.0988106333016 |pagamento       |2024-01-13 00:00:00  |approved|53.39574103171587 |false   |Standard |635         |2024|1    |13 |6          |medio      |
|31751         |6217       |73.90289198711338 |pagamento       |2024-0

In [7]:
df.createOrReplaceTempView("silver_transactions")

print("View silver_transactions criada.")

View silver_transactions criada.


In [8]:
resultado_sql = spark.sql("""
    SELECT
        segment,
        ROUND(AVG(credit_score), 1) AS score_medio,
        COUNT(*) AS total_transacoes
    FROM silver_transactions
    GROUP BY segment
    ORDER BY score_medio
""")

resultado_sql.show()

+---------+-----------+----------------+
|  segment|score_medio|total_transacoes|
+---------+-----------+----------------+
| Standard|      643.9|           29689|
|  Premium|      652.7|           61156|
|High-Risk|      659.8|            9155|
+---------+-----------+----------------+



In [9]:
resultado_dataframe = (
    df
    .groupBy("segment")
    .agg(
        F.round(
            F.avg("credit_score"),
            1
        ).alias("score_medio"),

        F.count("*").alias("total_transacoes")
    )
    .orderBy("score_medio")
)

resultado_dataframe.show()

+---------+-----------+----------------+
|  segment|score_medio|total_transacoes|
+---------+-----------+----------------+
| Standard|      643.9|           29689|
|  Premium|      652.7|           61156|
|High-Risk|      659.8|            9155|
+---------+-----------+----------------+



In [10]:
linhas_sql = resultado_sql.collect()
linhas_dataframe = resultado_dataframe.collect()

comparacao_metodos = [
    {
        "segment": linha_sql["segment"],
        "sql_score": linha_sql["score_medio"],
        "dataframe_score": linha_df["score_medio"],
        "sql_total": linha_sql["total_transacoes"],
        "dataframe_total": linha_df["total_transacoes"],
        "resultado_igual": linha_sql == linha_df
    }
    for linha_sql, linha_df
    in zip(linhas_sql, linhas_dataframe)
]

for resultado in comparacao_metodos:
    print(resultado)

{'segment': 'Standard', 'sql_score': 643.9, 'dataframe_score': 643.9, 'sql_total': 29689, 'dataframe_total': 29689, 'resultado_igual': True}
{'segment': 'Premium', 'sql_score': 652.7, 'dataframe_score': 652.7, 'sql_total': 61156, 'dataframe_total': 61156, 'resultado_igual': True}
{'segment': 'High-Risk', 'sql_score': 659.8, 'dataframe_score': 659.8, 'sql_total': 9155, 'dataframe_total': 9155, 'resultado_igual': True}


In [11]:
df.unpersist(blocking=True)

inicio_sem_cache = time.perf_counter()

fraudes_sem_cache = (
    df
    .filter(F.col("is_fraud") == True)
    .count()
)

tempo_sem_cache = time.perf_counter() - inicio_sem_cache

print("Fraudes:", fraudes_sem_cache)
print(f"Tempo sem cache: {tempo_sem_cache:.6f} segundos")

Fraudes: 1833
Tempo sem cache: 0.551212 segundos


In [12]:
df.cache()

inicio_materializacao = time.perf_counter()
total_cache = df.count()
tempo_materializacao = time.perf_counter() - inicio_materializacao

print("Linhas materializadas no cache:", total_cache)
print(
    f"Tempo para materializar o cache: "
    f"{tempo_materializacao:.6f} segundos"
)

Linhas materializadas no cache: 100000
Tempo para materializar o cache: 2.120148 segundos


In [13]:
inicio_com_cache = time.perf_counter()

fraudes_com_cache = (
    df
    .filter(F.col("is_fraud") == True)
    .count()
)

tempo_com_cache = time.perf_counter() - inicio_com_cache

print("Fraudes:", fraudes_com_cache)
print(f"Tempo com cache: {tempo_com_cache:.6f} segundos")

if tempo_com_cache > 0:
    print(
        "Razão sem cache/com cache:",
        round(tempo_sem_cache / tempo_com_cache, 2)
    )

Fraudes: 1833
Tempo com cache: 0.618765 segundos
Razão sem cache/com cache: 0.89


In [14]:
fraude_dia_semana = spark.sql("""
    SELECT
        DAYOFWEEK(transaction_timestamp) AS numero_dia_semana,

        CASE DAYOFWEEK(transaction_timestamp)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Segunda-feira'
            WHEN 3 THEN 'Terça-feira'
            WHEN 4 THEN 'Quarta-feira'
            WHEN 5 THEN 'Quinta-feira'
            WHEN 6 THEN 'Sexta-feira'
            WHEN 7 THEN 'Sábado'
        END AS dia_semana,

        COUNT(*) AS total_transacoes,

        SUM(
            CASE WHEN is_fraud THEN 1 ELSE 0 END
        ) AS fraudes,

        ROUND(
            100.0
            * SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS taxa_fraude_pct

    FROM silver_transactions

    GROUP BY
        DAYOFWEEK(transaction_timestamp),

        CASE DAYOFWEEK(transaction_timestamp)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Segunda-feira'
            WHEN 3 THEN 'Terça-feira'
            WHEN 4 THEN 'Quarta-feira'
            WHEN 5 THEN 'Quinta-feira'
            WHEN 6 THEN 'Sexta-feira'
            WHEN 7 THEN 'Sábado'
        END

    ORDER BY numero_dia_semana
""")

fraude_dia_semana.show()

+-----------------+-------------+----------------+-------+---------------+
|numero_dia_semana|   dia_semana|total_transacoes|fraudes|taxa_fraude_pct|
+-----------------+-------------+----------------+-------+---------------+
|                1|      Domingo|           14034|    257|           1.83|
|                2|Segunda-feira|           13940|    283|           2.03|
|                3|  Terça-feira|           14222|    262|           1.84|
|                4| Quarta-feira|           14338|    234|           1.63|
|                5| Quinta-feira|           14452|    263|           1.82|
|                6|  Sexta-feira|           14484|    261|           1.80|
|                7|       Sábado|           14530|    273|           1.88|
+-----------------+-------------+----------------+-------+---------------+



In [15]:
reconciliacao_semanal = (
    fraude_dia_semana
    .agg(
        F.sum("total_transacoes").alias("transacoes"),
        F.sum("fraudes").alias("fraudes")
    )
    .collect()[0]
)

print("Transações:", reconciliacao_semanal["transacoes"])
print("Fraudes:", reconciliacao_semanal["fraudes"])

Transações: 100000
Fraudes: 1833


In [16]:
df.unpersist(blocking=True)
spark.stop()

print("Cache liberado.")
print("Sessão Spark encerrada.")
print("Lab 10 executado com sucesso.")

Cache liberado.
Sessão Spark encerrada.
Lab 10 executado com sucesso.


## Conclusão

A camada Silver foi lida pelo PySpark em modo local, preservando as 100.000 transações e os tipos definidos no arquivo Parquet.

A agregação por segmento foi executada por Spark SQL e pela API de DataFrame. Os dois métodos produziram resultados idênticos, demonstrando que representam interfaces diferentes para o mesmo motor de processamento.

Também foi avaliado o uso de cache. A primeira ação exigiu a leitura do arquivo, enquanto as ações posteriores puderam utilizar os dados materializados em memória. Em bases pequenas, a diferença de tempo pode oscilar, mas o cache tende a ser útil quando o mesmo DataFrame é consultado repetidamente.

Por fim, as transações foram agregadas por dia da semana, e os totais foram reconciliados com a Silver, mantendo 100.000 transações e 1.833 fraudes.